# PIE's benchmark

Trains PWPA on PIE's benchmark of unseen knockdowns in seen contexts (Replogle–Nadig: K562, RPE1, HepG2, Jurkat) and scores it with PIE's metric code, next to the values in PIE's paper.

- Data and splits: PIE's processed arrays and its four published folds.
- Priors: PIE's five genetic knowledge sources, the same gene information PIE gets.
- One PWPA per fold and cell line, trained on that line's training knockdowns, with RFE choosing the priors.
- Scores: the mean over cell lines, then the mean and standard error over folds.

Writes `results/pie_benchmark_scores.tsv` and the main figure, `figures/main_figure.png`.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir

sys.path.append("../1.model")
import pwpa

In [ ]:
# PIE's Table 4 as published (mean and sd over folds; se = sd / 2) and PWPA's current scores (K031-pie-five)
scores = pd.read_csv("results/pie_benchmark_scores.tsv", sep="\t")
scores.pivot(index="model", columns="metric", values="mean")

In [ ]:
# TODO(orcd): port immune_grn/lab/exp/K031-pie-five-path (training) and K020-pie-score/score.py (PIE's metric code,
# ArcInstitute/pie @ 21a526a). The current PWPA rows come from K031's L1 + decoys fit; rerunning here with RFE
# replaces them.
#
# priors = pwpa.load_priors(data_dir("priors"))  # PIE's five genetic knowledge sources
# rows = []
# for fold in range(4):
#     for line in ["k562", "rpe1", "hepg2", "jurkat"]:
#         train, test = PIE's split (fold, line) from data_dir("pie_benchmark")
#         best = pwpa.choose(pwpa.rfe(priors, [(train.genes, train.delta)]))
#         chosen = [priors[name] for name in best["priors"]]
#         pred, W = pwpa.predict(chosen, best["scale"], best["gates"], test.genes, train.genes, train.delta)
#         p_de, lfc = pwpa.predict_de(W, train.de, train.lfc, own=train.own_column)
#         rows += PIE's metrics of (pred, p_de, lfc) against test, per knockdown
# scores = mean over lines, then mean and SE over folds, in tidy long form (model, metric, mean, se)
# overwrite the pwpa rows of results/pie_benchmark_scores.tsv; keep PIE's published rows

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# the main figure: (A) how PWPA is trained, from 1.model; (B) PWPA against PIE's published results
METRICS = {
    "binary_auprc": "AUPRC",
    "sig_jaccard": "Jaccard",
    "direction_match": "DE direction match",
    "spearman_nsig": "Spearman effect size",
    "spearman_lfc": "Spearman LFC",
    "discrimination_score_l1": "Discrimination L1",
}
MODELS = {
    "context_mean": ("Context\nmean", "#8d97a8"),
    "compass_n": ("COMPASS-N", "#c9822f"),
    "presage": ("PRESAGE", "#7d9a4a"),
    "tabicl": ("TabICL", "#3f9c93"),
    "pie": ("PIE", "#2c6aa8"),
    "pwpa": ("PWPA", "#a03a8c"),
}
plt.rcParams.update({"font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold"})

fig = plt.figure(figsize=(12, 16), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=[7.4, 8.6])
diagram = top.subplots()
diagram.imshow(mpimg.imread("../1.model/figures/rfe_training.png"))
diagram.axis("off")
top.text(0.0, 1.0, "A", fontsize=20, fontweight="bold", va="top")

bottom.suptitle("PWPA and the models in PIE's paper: unseen knockdowns in seen contexts (Replogle–Nadig)", fontweight="bold")
bottom.text(0.0, 1.0, "B", fontsize=20, fontweight="bold", va="top")
for ax, (metric, title) in zip(bottom.subplots(2, 3).flat, METRICS.items()):
    cell = scores.query("metric == @metric").set_index("model")
    for x, (model, (_, color)) in enumerate(MODELS.items()):
        mean, se = cell.loc[model, ["mean", "se"]]
        if pd.isna(mean):  # the context mean has no Spearman effect size
            ax.text(x, 0.01, "NaN", ha="center", va="bottom", fontsize=8, color="#5a6478")
            continue
        ax.bar(x, mean, yerr=se, color=color, width=0.72, zorder=2, error_kw={"lw": 1, "capsize": 3})
        ax.text(x, mean + se, f"{mean:.3f}", ha="center", va="bottom", fontsize=7.5, family="monospace")
    ax.set_title(title)
    ax.set_xticks(range(len(MODELS)), [name.replace("\n", " ") for name, _ in MODELS.values()], rotation=35, ha="right", fontsize=8.5)
    ax.get_xticklabels()[-1].set_fontweight("bold")
    ax.grid(axis="y", color="#e3e6ec", zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(-0.6, len(MODELS) - 0.4)
fig.savefig("figures/main_figure.png", dpi=200, bbox_inches="tight", facecolor="white")